In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv") #-data.csv")

In [2]:
df_ground_truth.head()

,question,document
0,I just found this course — is it too late to j...,74eb249bbf
1,Can I still start the course if I'm coming in ...,74eb249bbf
2,"If I enroll now, is there any chance to get a ...",74eb249bbf
3,What do I need to do to be eligible for the co...,74eb249bbf
4,Is the final project deadline the only thing t...,74eb249bbf


In [3]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
ground_truth[10]

{'question': 'How do students join the Office Hours or live workshop sessions if the Zoom link isn’t public?',
 'document': '489dd1c9d9'}

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()


In [6]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [7]:
boost = {'question': 3.0}

index.search(
    "What is the course about?",
    num_results=5,
    boost_dict=boost
)

[{'id': 'db78580409',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'What is the cosine similarity?',
  'answer': 'Cosine similarity is a measure used to calculate the similarity between two non-zero vectors, often used in text analysis to determine how similar two documents are based on their content. This metric computes the cosine of the angle between two vectors, which are typically word counts or TF-IDF values of the documents. The cosine similarity value ranges from -1 to 1, where 1 indicates that the vectors are identical, 0 indicates that the vectors are orthogonal (no similarity), and -1 represents completely opposite vectors.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](h

In [8]:
# Функция-обёртка поиска с фиксированными весами полей — используем её как "текстовый поиск v1"
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}   # question важнее, section — немного важнее базового веса

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [9]:
# Берём первый вопрос из ground truth для проверки поиска
q = ground_truth[0]
q

{'question': 'I just found this course — is it too late to join now?',
 'document': '74eb249bbf'}

In [10]:
# id документа, который считается ПРАВИЛЬНЫМ (эталонным) ответом на этот вопрос
doc_id = q['document']
doc_id

'74eb249bbf'

In [11]:
# Выполняем поиск по тексту вопроса и смотрим, что вернул индекс
results = text_search(q['question'])
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in order to complete the course homeworks. You can use free or low-cost alternatives listed in the course GitHub repo.\n\nSee the course list of [OpenAI API alternatives](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/awesome-llms.md#openai-api-alternatives)."},
 {'id': '0fab61eca2',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'Is it a group project?',
  'answer': 'No, the capstone is an individual project.\n\nYou can collaborate o

In [12]:
# Проверяем, для каждого найденного документа: совпадает ли его id с эталонным doc_id
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
85384a18e5 == 74eb249bbf: False
0fab61eca2 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
86d99bbf21 == 74eb249bbf: False


In [13]:
# Строим бинарный вектор релевантности: 1, если найденный документ — тот самый эталонный, иначе 0
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [14]:
# Функция: считает вектор релевантности для одного вопроса ground truth через text_search
def compute_relevance_text(q):
    doc_id = q["document"]                 # эталонный id документа для этого вопроса
    results = text_search(query=q["question"])  # результаты поиска по вопросу

    relevance = []
    for d in results:                        # для каждого найденного документа
        relevance.append(int(d["id"] == doc_id))  # 1 — если это эталонный документ, 0 — если нет

    return relevance

In [15]:
# Проверяем функцию на первом вопросе — ожидаем, что эталонный документ найдётся на 1-й позиции
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I just found this course — is it too late to join now?


[1, 0, 0, 0, 0]

In [16]:
# Проверяем на 12-м вопросе — эталонный документ найден на 3-й позиции
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [0, 0, 1, 0, 0]

Where can I find the live stream link for office hours before the session starts?


[1, 0, 0, 0, 0]

In [17]:
# Ожидаемый результат предыдущей ячейки (для сравнения при чтении ноутбука)
[0, 0, 1, 0, 0]

[0, 0, 1, 0, 0]

In [18]:
# Проверяем на 51-м вопросе — эталонный документ вообще НЕ найден среди top-5
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [0, 0, 0, 0, 0]

Where should I check the LLM Zoomcamp syllabus, homework deadlines, and my progress for the current cohort?


[1, 0, 0, 0, 0]

In [19]:
# Ожидаемый результат предыдущей ячейки
[0, 0, 0, 0, 0]

[0, 0, 0, 0, 0]

In [20]:
# Функция: считает векторы релевантности для ВСЕХ вопросов ground truth (текстовый поиск)
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []                      # список векторов релевантности, по одному на вопрос

    for q in tqdm(ground_truth):              # прогресс-бар по всем вопросам
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [21]:
# Считаем релевантность для всего ground truth-датасета
relevance = compute_relevance_total_text(ground_truth)

  0%|          | 0/560 [00:00<?, ?it/s]

In [22]:
# Смотрим на первые 15 векторов релевантности
relevance[:15]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [23]:
# Обобщённая версия compute_relevance: принимает ЛЮБУЮ функцию поиска (не только text_search),
# что позволяет сравнивать разные стратегии поиска (разные веса, разные алгоритмы и т.д.)
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])   # поиск через переданную функцию

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [24]:
# Обобщённая версия compute_relevance_total: тоже принимает функцию поиска параметром
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [25]:
# Считаем релевантность для всего датасета через обобщённую функцию (результат совпадает с ячейкой 19)
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/560 [00:00<?, ?it/s]

In [26]:
# Берём небольшую выборку (первые 15 записей) для ручной проверки метрик перед тем, как считать их на всём датасете
sample = relevance_total[:15]

In [27]:
# Смотрим на выборку векторов релевантности
sample

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [28]:
# "Ручной" расчёт hit rate на глаз: 14 вопросов из 15 нашли эталонный документ хотя бы в top-5
14 / 15

0.9333333333333333

In [29]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

0.7333333333333333

In [30]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [31]:
hit_rate(relevance)

0.8392857142857143

In [32]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            score = 1 / (rank + 1)
            total_score = total_score + score
            break

total_score / len(sample)

0.6555555555555554

In [33]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [34]:
mrr(sample)

0.6555555555555554

In [38]:
mrr(relevance)

0.7149107142857141

In [39]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [40]:
evaluate(ground_truth, text_search)

  0%|          | 0/560 [00:00<?, ?it/s]

{'hit_rate': 0.8392857142857143, 'mrr': 0.7149107142857141}

In [41]:
def text_search_v2(query):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [42]:
evaluate(ground_truth, text_search_v2)

  0%|          | 0/560 [00:00<?, ?it/s]

{'hit_rate': 0.8696428571428572, 'mrr': 0.7407738095238091}

In [43]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [44]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/560 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8946428571428572, 'mrr': 0.7825595238095235}


  0%|          | 0/560 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.9035714285714286, 'mrr': 0.7754761904761898}


  0%|          | 0/560 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8392857142857143, 'mrr': 0.7149107142857141}


  0%|          | 0/560 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8160714285714286, 'mrr': 0.6869047619047617}


  0%|          | 0/560 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7875, 'mrr': 0.6573214285714282}


In [45]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [46]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(f"Evaluating question_boost={question_boost}, answer_boost={answer_boost}, section_boost={section_boost}...")
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/560 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/560 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/560 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/560 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/560 [00:00<?, ?it/s]

In [47]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
8,1.0,4.0,0.5,0.962500,0.863393
4,1.0,2.0,0.2,0.964286,0.863393
7,1.0,4.0,0.2,0.964286,0.862619
20,2.0,4.0,0.5,0.964286,0.862560
35,5.0,10.0,0.5,0.969643,0.862381
19,2.0,4.0,0.2,0.969643,0.862381
3,1.0,2.0,0.1,0.969643,0.862381
6,1.0,4.0,0.1,0.962500,0.860327
18,2.0,4.0,0.1,0.969643,0.860000
34,5.0,10.0,0.2,0.969643,0.859613
